In [ ]:
!pip -q install -U "transformers>=4.46.0" "peft>=0.13.0" "accelerate>=1.0.0" "torchao>=0.16.0" scikit-learn safetensors tqdm

from google.colab import drive
drive.mount('/content/drive')

import os, math, random
from copy import deepcopy
from collections import defaultdict, Counter
from itertools import product

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler

from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json, csv, time, gc

TEST_ROOT = "/content/drive/MyDrive/Test"
SUITE_DIR = os.path.join(TEST_ROOT, "Python", "test_bridge_simple")
os.makedirs(SUITE_DIR, exist_ok=True)

MODEL_NAME = "EleutherAI/pythia-1.4b-deduped"
SEED = 42
DATA_SEED = 42
SPLIT_SEED = 42
TRAIN_SEED_BASE = 42
EPOCHS = 15
TRAIN_FRAC = 0.85

BATCH_SIZE = 16
LR = 5e-5
MAX_LENGTH = 256

BASE_LAMBDA = 1000.0

WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
GRAD_ACCUM = 1
MAX_GRAD_NORM = 1.0

INPUT_BLOCK_EXTRAS = 2

USE_EXTRA_ROWS = True

LORA_R = 64
LORA_ALPHA = 64
LORA_DROPOUT = 0.10
LORA_TARGET_MODULES = ["query_key_value", "dense", "dense_h_to_4h", "dense_4h_to_h"]

VARS = ["a", "b", "c", "d"]
OPS = ["or", "and", "iff"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

VARIANTS = [
    {
        "idx": 0,
        "name": "baseline",
        "full_prompt_rows": 0,
        "format_row_weight": 1,
        "contrastive_scale": 1.0,
        "temperature": 0.05,
        "pooling": "last",
    },
    {
        "idx": 1,
        "name": "full_prompts_50",
        "full_prompt_rows": 50,
        "format_row_weight": 1,
        "contrastive_scale": 1.0,
        "temperature": 0.05,
        "pooling": "last",
    },
    {
        "idx": 3,
        "name": "format_2x",
        "full_prompt_rows": 0,
        "format_row_weight": 2,
        "contrastive_scale": 1.0,
        "temperature": 0.05,
        "pooling": "last",
    },
    {
        "idx": 4,
        "name": "full_prompts_50_format_2x",
        "full_prompt_rows": 50,
        "format_row_weight": 2,
        "contrastive_scale": 1.0,
        "temperature": 0.05,
        "pooling": "last",
    },
    {
        "idx": 5,
        "name": "contrastive_0p5x",
        "full_prompt_rows": 0,
        "format_row_weight": 1,
        "contrastive_scale": 0.5,
        "temperature": 0.05,
        "pooling": "last",
    },
    {
        "idx": 6,
        "name": "contrastive_2x",
        "full_prompt_rows": 0,
        "format_row_weight": 1,
        "contrastive_scale": 2.0,
        "temperature": 0.05,
        "pooling": "last",
    },
    {
        "idx": 9,
        "name": "mean_pooling",
        "full_prompt_rows": 0,
        "format_row_weight": 1,
        "contrastive_scale": 1.0,
        "temperature": 0.05,
        "pooling": "mean",
    },
]

RUN_IDS = {0, 1, 3, 4, 5, 6, 9}
RESUME_SKIP_COMPLETED = False
PRINT_FIRST_DETAILS = False

print("SUITE_DIR:", SUITE_DIR)
print("DEVICE:", DEVICE)
print("variants:", [v["name"] for v in VARIANTS if v["idx"] in RUN_IDS])
print("shared settings:", {
    "epochs": EPOCHS,
    "lr": LR,
    "batch_size": BATCH_SIZE,
    "max_length": MAX_LENGTH,
    "base_contrastive_lambda": BASE_LAMBDA,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "cross_n_input_block_extras": INPUT_BLOCK_EXTRAS,
    "use_role_confusion_rows": USE_EXTRA_ROWS,
    "data_seed": DATA_SEED,
    "split_seed": SPLIT_SEED,
    "train_seed_base": TRAIN_SEED_BASE,
})

def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def bool_word(x):
    return "True" if x else "False"

def hamming(xs, ys):
    return sum(int(a != b) for a, b in zip(xs, ys))

def masked_mean(x, mask):
    if mask.any():
        return x[mask].mean().item()
    return float("nan")

def var_list(n):
    return VARS[:n]

PERSON_BY_VAR = {
    "a": "Eyal",
    "b": "Shahar",
    "c": "Noa",
    "d": "Shelley",
}
PERSON_ORDER = ["Eyal", "Shahar", "Noa", "Shelley"]

def person_list(n):
    return PERSON_ORDER[:n]

def person_status(name, val):
    return f"{name} is truthful." if val else f"{name} is lying."

def Atom(v, neg=False):
    return ("atom", v, bool(neg))

def Bin(op, left_atom, right_atom):
    return ("bin", op, left_atom, right_atom)

def atom_key(atom):
    _, v, neg = atom
    return f"not_{v}" if neg else v

def lit_sort_key(atom):
    _, v, neg = atom
    return (v, int(neg))

def expr_surface_id(expr):
    if expr[0] == "atom":
        return atom_key(expr)
    _, op, a1, a2 = expr
    return f"{op}({atom_key(a1)},{atom_key(a2)})"

def expr_vars(expr):
    if expr[0] == "atom":
        return (expr[1],)
    _, _, a1, a2 = expr
    return tuple(sorted({a1[1], a2[1]}))

def eval_atom(atom, env):
    _, v, neg = atom
    val = bool(env[v])
    return (not val) if neg else val

def eval_expr(expr, env):
    if expr[0] == "atom":
        return eval_atom(expr, env)

    _, op, a1, a2 = expr
    x = eval_atom(a1, env)
    y = eval_atom(a2, env)

    if op == "and":
        return x and y
    if op == "or":
        return x or y
    if op == "iff":
        return x == y
    raise ValueError(op)

def expr_truth_table(expr):
    vars_ = expr_vars(expr)
    bits = []
    for vals in product([False, True], repeat=len(vars_)):
        env = dict(zip(vars_, vals))
        bits.append("1" if eval_expr(expr, env) else "0")
    return vars_, "".join(bits)

def expr_canonical_id(expr):
    return f"surface|{expr_surface_id(expr)}"

def expr_equiv_id(expr):
    vars_, table = expr_truth_table(expr)
    return f"truth|vars={','.join(vars_)}|tt={table}"

def render_atom(atom, surface):
    _, v, neg = atom
    if surface == "python":
        return f"(not {v})" if neg else v
    name = PERSON_BY_VAR[v]
    return f"{name} is lying" if neg else f"{name} is truthful"

def render_expr(expr, surface):
    if expr[0] == "atom":
        return render_atom(expr, surface)

    _, op, a1, a2 = expr
    left = render_atom(a1, surface)
    right = render_atom(a2, surface)

    if surface == "python":
        if op == "or":
            return f"{left} or {right}"
        if op == "and":
            return f"{left} and {right}"
        if op == "iff":
            return f"{left} == {right}"
        raise ValueError(op)

    if op == "or":
        return f"at least one of the following is True: {left}; {right}"
    if op == "and":
        return f"both of the following are True: {left}; {right}"
    if op == "iff":
        return f"{left} exactly when {right}"
    raise ValueError(op)

def make_text(expr, surface, text_mode, speaker_idx=0):
    body = render_expr(expr, surface)
    if text_mode == "expr":
        return body
    if text_mode == "rule":
        if surface == "python":
            return f"    rule_{speaker_idx + 1} = {body}"
        return f"{PERSON_ORDER[speaker_idx]} says: {body}"
    raise ValueError(text_mode)

def build_atoms(vars_):
    out = []
    for v in vars_:
        out.append(Atom(v, False))
        out.append(Atom(v, True))
    return out

def build_exprs():
    atoms = build_atoms(VARS)
    exprs = []
    exprs.extend(atoms)

    for op in OPS:
        for a1 in atoms:
            for a2 in atoms:
                exprs.append(Bin(op, a1, a2))

    seen = set()
    deduped = []
    for e in exprs:
        sid = expr_surface_id(e)
        if sid not in seen:
            seen.add(sid)
            deduped.append(e)
    return deduped

def expr_meta(expr):
    vars_used, truth_table = expr_truth_table(expr)
    canonical = expr_canonical_id(expr)
    equiv = expr_equiv_id(expr)

    if expr[0] == "atom":
        _, v, neg = expr
        return {
            "kind": "atom",
            "var": v,
            "neg": int(neg),
            "vars_used": vars_used,
            "truth_table": truth_table,
            "op": None,
            "same_variable": 0,
            "surface": expr_surface_id(expr),
            "canonical": canonical,
            "equiv": equiv,
        }

    _, op, a1, a2 = expr
    ordered_lits = ((a1[1], int(a1[2])), (a2[1], int(a2[2])))
    return {
        "kind": "bin",
        "op": op,
        "vars_ordered": tuple(v for v, _ in ordered_lits),
        "negs_ordered": tuple(n for _, n in ordered_lits),
        "vars_used": vars_used,
        "truth_table": truth_table,
        "same_variable": int(a1[1] == a2[1]),
        "surface": expr_surface_id(expr),
        "canonical": canonical,
        "equiv": equiv,
    }

def batch_bucket_key(meta):
    if meta["kind"] == "format":
        ft = meta["format_type"]
        if ft in {"input_block", "full_prompt", "full_prompt_val"}:
            return ("format", ft, meta["n"])
        return ("format", ft)

    if meta["kind"] == "atom":
        return ("logic", meta["text_mode"], "atom", meta["var"])

    if meta["kind"] == "bin" and meta.get("same_variable") == 1:
        return ("logic", meta["text_mode"], "samevar", meta["op"], meta["vars_ordered"][0])

    return ("logic", meta["text_mode"], "bin_ordered_vars", meta["vars_ordered"])

def contrastive_ok_from_meta(meta):
    return True

def add_row(rows, semantic_id, canonical_id, equiv_id, text_en, text_py, meta):
    meta = dict(meta)
    contrastive_ok = contrastive_ok_from_meta(meta)
    rows.append({
        "semantic_id": semantic_id,
        "canonical_id": canonical_id,
        "equiv_id": equiv_id,
        "text_en": text_en,
        "text_py": text_py,
        "meta": meta,
        "bucket_key": batch_bucket_key(meta),
        "contrastive_ok": contrastive_ok,
    })

def build_format_rows():
    rows = []

    for n in [2, 3, 4]:
        vars_ = var_list(n)
        people = person_list(n)
        sid = f"format|header|n={n}"
        add_row(
            rows,
            semantic_id=sid,
            canonical_id=sid,
            equiv_id=sid,
            text_py=f"def check({', '.join(vars_)}) -> bool:",
            text_en=f"Given labels for {', '.join(people)}, determine whether the assignment is True or False.",
            meta={"kind": "format", "text_mode": "format", "format_type": "header", "n": n},
        )

    for i, (v, name) in enumerate(zip(VARS, PERSON_ORDER)):
        sid = f"format|return_condition|idx={i}"
        add_row(
            rows,
            semantic_id=sid,
            canonical_id=sid,
            equiv_id=sid,
            text_py=f"{v} == rule_{i + 1}",
            text_en=f"{name} is labeled truthful exactly when {name}'s statement is truthful.",
            meta={"kind": "format", "text_mode": "format", "format_type": "return_condition", "idx": i},
        )

    for n in [2, 3, 4]:
        vars_ = var_list(n)
        people = person_list(n)
        text_py = "    return " + " and ".join(f"{v} == rule_{i + 1}" for i, v in enumerate(vars_))
        en_lines = ["Determine whether all of the following hold:"]
        for name in people:
            en_lines.append(f"{name} is labeled truthful exactly when {name}'s statement is truthful.")
        sid = f"format|return_block|n={n}"
        add_row(
            rows,
            semantic_id=sid,
            canonical_id=sid,
            equiv_id=sid,
            text_py=text_py,
            text_en="\n".join(en_lines),
            meta={"kind": "format", "text_mode": "format", "format_type": "return_block", "n": n},
        )

    for i, (v, name) in enumerate(zip(VARS, PERSON_ORDER)):
        for val in [False, True]:
            sid = f"format|input_line|idx={i}|val={int(val)}"
            add_row(
                rows,
                semantic_id=sid,
                canonical_id=sid,
                equiv_id=sid,
                text_py=f"{v} = {bool_word(val)}",
                text_en=person_status(name, val),
                meta={"kind": "format", "text_mode": "format", "format_type": "input_line", "idx": i, "val": int(val)},
            )

    for n in [2, 3, 4]:
        vars_ = var_list(n)
        people = person_list(n)
        for assign in product([False, True], repeat=n):
            key = "".join("1" if x else "0" for x in assign)
            py_lines = ["Input:"]
            en_lines = ["Input:"]
            for v, name, val in zip(vars_, people, assign):
                py_lines.append(f"{v} = {bool_word(val)}")
                en_lines.append(person_status(name, val))
            sid = f"format|input_block|n={n}|assign={key}"
            add_row(
                rows,
                semantic_id=sid,
                canonical_id=sid,
                equiv_id=sid,
                text_py="\n".join(py_lines),
                text_en="\n".join(en_lines),
                meta={"kind": "format", "text_mode": "format", "format_type": "input_block", "n": n, "assign": key},
            )

    return rows

def expr_allowed_for_n(expr, n):
    return set(expr_vars(expr)).issubset(set(var_list(n)))

def render_full_prompt(n, rule_exprs, assign):
    vars_ = var_list(n)
    people = person_list(n)

    py_lines = [f"def check({', '.join(vars_)}) -> bool:", ""]
    for i, expr in enumerate(rule_exprs):
        py_lines.append(make_text(expr, "python", "rule", i))
    py_lines.append("")
    py_lines.append("    " + "return " + " and ".join(f"{v} == rule_{i + 1}" for i, v in enumerate(vars_)))
    py_lines.append("")
    py_lines.append("Input:")
    for v, val in zip(vars_, assign):
        py_lines.append(f"{v} = {bool_word(val)}")
    py_lines.append("")
    py_lines.append("Output:")

    en_lines = [f"Given labels for {', '.join(people)}, determine whether the assignment is True or False.", ""]
    for i, expr in enumerate(rule_exprs):
        en_lines.append(make_text(expr, "english", "rule", i))
    en_lines.append("")
    en_lines.append("Determine whether all of the following hold:")
    for name in people:
        en_lines.append(f"{name} is labeled truthful exactly when {name}'s statement is truthful.")
    en_lines.append("")
    en_lines.append("Input:")
    for name, val in zip(people, assign):
        en_lines.append(person_status(name, val))
    en_lines.append("")
    en_lines.append("Output:")

    return "\n".join(py_lines), "\n".join(en_lines)

def build_full_prompt_rows(num_rows, seed):
    if num_rows <= 0:
        return []

    rng = random.Random(seed + 10000)
    rows = []
    exprs_all = build_exprs()
    exprs_by_n = {
        n: [e for e in exprs_all if expr_allowed_for_n(e, n)]
        for n in [2, 3, 4]
    }

    seen = set()
    attempts = 0
    while len(rows) < num_rows and attempts < num_rows * 200:
        attempts += 1
        n = rng.choice([2, 3, 4])
        exprs_n = exprs_by_n[n]
        rule_exprs = [rng.choice(exprs_n) for _ in range(n)]
        assign = tuple(rng.choice([False, True]) for _ in range(n))
        assign_key = "".join("1" if x else "0" for x in assign)

        surface_parts = [expr_surface_id(e) for e in rule_exprs]
        canon_parts = [expr_canonical_id(e) for e in rule_exprs]
        equiv_parts = [expr_equiv_id(e) for e in rule_exprs]

        semantic_id = f"full_prompt|n={n}|rules={'/'.join(surface_parts)}|assign={assign_key}"
        if semantic_id in seen:
            continue
        seen.add(semantic_id)

        canonical_id = f"full_prompt|n={n}|rules={'/'.join(canon_parts)}|assign={assign_key}"
        equiv_id = f"full_prompt|n={n}|rules={'/'.join(equiv_parts)}|assign={assign_key}"

        text_py, text_en = render_full_prompt(n, rule_exprs, assign)
        add_row(
            rows,
            semantic_id=semantic_id,
            canonical_id=canonical_id,
            equiv_id=equiv_id,
            text_en=text_en,
            text_py=text_py,
            meta={
                "kind": "format",
                "text_mode": "format",
                "format_type": "full_prompt",
                "n": n,
                "assign": assign_key,
            },
        )

    assert len(rows) == num_rows, f"Could only build {len(rows)} full prompt rows out of requested {num_rows}"
    return rows

def build_full_prompt_val_rows():
    specs = [
        (2, [Atom("a"), Atom("b", True)], (True, False)),
        (3, [
            Bin("and", Atom("a"), Atom("b")),
            Bin("or", Atom("b", True), Atom("c")),
            Bin("iff", Atom("a"), Atom("c", True)),
        ], (True, False, True)),
        (4, [
            Bin("iff", Atom("a"), Atom("b")),
            Bin("and", Atom("c"), Atom("d", True)),
            Bin("or", Atom("a", True), Atom("d")),
            Atom("c", True),
        ], (False, True, False, True)),
    ]

    rows = []
    for case_idx, (n, rule_exprs, assign) in enumerate(specs):
        assign_key = "".join("1" if x else "0" for x in assign)
        surface_parts = [expr_surface_id(e) for e in rule_exprs]
        canon_parts = [expr_canonical_id(e) for e in rule_exprs]
        equiv_parts = [expr_equiv_id(e) for e in rule_exprs]

        sid = f"format|full_prompt_val|n={n}|case={case_idx}"
        canonical_id = f"full_prompt|n={n}|rules={'/'.join(canon_parts)}|assign={assign_key}"
        equiv_id = f"full_prompt|n={n}|rules={'/'.join(equiv_parts)}|assign={assign_key}"

        text_py, text_en = render_full_prompt(n, rule_exprs, assign)
        add_row(
            rows,
            semantic_id=sid,
            canonical_id=canonical_id,
            equiv_id=equiv_id,
            text_en=text_en,
            text_py=text_py,
            meta={"kind": "format", "text_mode": "format", "format_type": "full_prompt_val", "n": n, "case": case_idx},
        )
    return rows

def build_rows(cfg=None):
    cfg = cfg or {}
    rows = []
    exprs = build_exprs()

    for expr in exprs:
        surface = expr_surface_id(expr)
        canonical = expr_canonical_id(expr)
        meta0 = expr_meta(expr)

        meta = {**meta0, "text_mode": "expr"}
        add_row(
            rows,
            semantic_id=f"expr|{surface}",
            canonical_id=f"expr|{canonical}",
            equiv_id=f"expr|{meta0['equiv']}",
            text_en=make_text(expr, "english", "expr"),
            text_py=make_text(expr, "python", "expr"),
            meta=meta,
        )

        for speaker_idx in range(len(VARS)):
            meta = {**meta0, "text_mode": "rule", "speaker_idx": speaker_idx}
            add_row(
                rows,
                semantic_id=f"rule|{surface}|speaker={speaker_idx}",
                canonical_id=f"rule|{canonical}|speaker={speaker_idx}",
                equiv_id=f"rule|{meta0['equiv']}|speaker={speaker_idx}",
                text_en=make_text(expr, "english", "rule", speaker_idx),
                text_py=make_text(expr, "python", "rule", speaker_idx),
                meta=meta,
            )

    rows.extend(build_format_rows())

    full_prompt_rows = int(cfg.get("full_prompt_rows", 0))
    if full_prompt_rows > 0:
        rows.extend(build_full_prompt_rows(full_prompt_rows, int(cfg.get("seed", SEED))))

    return rows

def build_role_confusion_views(source_rows):
    by_semantic = {}
    for r in source_rows:
        by_semantic.setdefault(r["semantic_id"], r)

    confusion_rows = []
    seen = set()

    def atom_sem(v, neg=False):
        return f"expr|{expr_surface_id(Atom(v, neg))}"

    def rule_atom_sem(v, neg, speaker_idx):
        return f"rule|{expr_surface_id(Atom(v, neg))}|speaker={speaker_idx}"

    def input_line_sem(idx, val):
        return f"format|input_line|idx={idx}|val={int(val)}"

    def return_condition_sem(idx):
        return f"format|return_condition|idx={idx}"

    def iff_expr_sem(v, w):
        return f"expr|{expr_surface_id(Bin('iff', Atom(v, False), Atom(w, False)))}"

    def add_view(semantic_id, confusion_type, bucket_parts):
        base = by_semantic.get(semantic_id)
        if base is None:
            return
        bucket_key = ("role_confusion", confusion_type) + tuple(bucket_parts)
        key = (semantic_id, bucket_key)
        if key in seen:
            return
        seen.add(key)

        rr = deepcopy(base)
        rr["meta"] = dict(rr["meta"])
        rr["meta"]["role_confusion_view"] = 1
        rr["meta"]["confusion_type"] = confusion_type
        rr["bucket_key"] = bucket_key
        rr["contrastive_ok"] = True
        confusion_rows.append(rr)

    def add_bucket(semantic_ids, confusion_type, bucket_parts):
        before = len(confusion_rows)
        for sid in semantic_ids:
            add_view(sid, confusion_type, bucket_parts)

        added = confusion_rows[before:]
        if len({r["canonical_id"] for r in added}) < 2:
            del confusion_rows[before:]

    for idx, v in enumerate(VARS):
        add_bucket(
            [input_line_sem(idx, True), atom_sem(v, False)],
            "input_vs_atom",
            (v, "true"),
        )
        add_bucket(
            [input_line_sem(idx, False), atom_sem(v, True)],
            "input_vs_atom",
            (v, "false"),
        )

    for idx, v in enumerate(VARS):
        for speaker_idx in range(len(VARS)):
            add_bucket(
                [input_line_sem(idx, True), rule_atom_sem(v, False, speaker_idx)],
                "input_vs_rule_atom",
                (v, "true", f"speaker={speaker_idx}"),
            )
            add_bucket(
                [input_line_sem(idx, False), rule_atom_sem(v, True, speaker_idx)],
                "input_vs_rule_atom",
                (v, "false", f"speaker={speaker_idx}"),
            )

    for v in VARS:
        for neg in [False, True]:
            sign = "neg" if neg else "pos"
            for speaker_idx in range(len(VARS)):
                add_bucket(
                    [atom_sem(v, neg), rule_atom_sem(v, neg, speaker_idx)],
                    "expr_vs_rule_atom",
                    (v, sign, f"speaker={speaker_idx}"),
                )

    for idx, v in enumerate(VARS):
        add_bucket(
            [
                return_condition_sem(idx),
                rule_atom_sem(v, False, idx),
                rule_atom_sem(v, True, idx),
            ],
            "rule_atom_vs_return_condition",
            (v, f"rule={idx + 1}"),
        )

    for idx, v in enumerate(VARS):
        ids = [return_condition_sem(idx)]
        ids.extend(iff_expr_sem(v, w) for w in VARS)
        add_bucket(
            ids,
            "iff_expr_vs_return_condition",
            (v, f"return={idx + 1}"),
        )

    for i in range(len(VARS)):
        for j in range(i + 1, len(VARS)):
            vi, vj = VARS[i], VARS[j]
            for neg in [False, True]:
                sign = "neg" if neg else "pos"
                add_bucket(
                    [
                        rule_atom_sem(vj, neg, i),
                        rule_atom_sem(vi, neg, j),
                    ],
                    "speaker_content_swap",
                    (f"speaker={i}_content={vj}", f"speaker={j}_content={vi}", sign),
                )

    return confusion_rows

def important_format_val_ids(rows):
    out = set()
    keep_input_lines = {
        (0, 0), (0, 1),
        (1, 0), (1, 1),
    }
    keep_input_blocks = {
        (2, "01"),
        (3, "010"),
        (4, "0101"),
        (4, "0000"),
        (4, "1111"),
    }

    for r in rows:
        m = r["meta"]
        if m.get("kind") != "format":
            continue

        ft = m.get("format_type")
        if ft == "header" and m.get("n") == 3:
            out.add(r["canonical_id"])
        elif ft == "return_block" and m.get("n") == 3:
            out.add(r["canonical_id"])
        elif ft == "return_condition" and m.get("idx") in {0, 2}:
            out.add(r["canonical_id"])
        elif ft == "input_line" and (m.get("idx"), m.get("val")) in keep_input_lines:
            out.add(r["canonical_id"])
        elif ft == "input_block" and (m.get("n"), m.get("assign")) in keep_input_blocks:
            out.add(r["canonical_id"])
    return out

def split_rows(rows, seed=SEED, force_val_ids=None):
    force_val_ids = set(force_val_ids or [])
    format_ids = {r["canonical_id"] for r in rows if r["meta"].get("kind") == "format"}
    random_ids = sorted({
        r["canonical_id"] for r in rows
        if r["canonical_id"] not in force_val_ids and r["canonical_id"] not in format_ids
    })
    rng = random.Random(seed)
    rng.shuffle(random_ids)
    cut = max(1, int(len(random_ids) * TRAIN_FRAC))
    train_ids = set(random_ids[:cut]) | (format_ids - force_val_ids)
    val_ids = set(random_ids[cut:]) | force_val_ids
    return [r for r in rows if r["canonical_id"] in train_ids], [r for r in rows if r["canonical_id"] in val_ids]

class PairDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, idx):
        return self.rows[idx]

class PairCollator:
    def __init__(self, tokenizer):
        self.tok = tokenizer

    def tokenize(self, texts):
        enc = self.tok(
            texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        labels = enc["input_ids"].clone()
        labels[enc["attention_mask"] == 0] = -100
        enc["labels"] = labels
        return enc

    def __call__(self, batch):
        en = [x["text_en"] for x in batch]
        py = [x["text_py"] for x in batch]
        enc_en = self.tokenize(en)
        enc_py = self.tokenize(py)
        return {
            "en": enc_en,
            "py": enc_py,
            "semantic_ids": [x["semantic_id"] for x in batch],
            "canonical_ids": [x["canonical_id"] for x in batch],
            "equiv_ids": [x["equiv_id"] for x in batch],
            "bucket_keys": [x["bucket_key"] for x in batch],
            "contrastive_ok": [x["contrastive_ok"] for x in batch],
        }

class PairBatchSampler(Sampler):
    def __init__(self, rows, batch_size, seed):
        self.rows = rows
        self.batch_size = batch_size
        self.seed = seed
        self.epoch = 0
        self.by_bucket = defaultdict(list)
        for idx, r in enumerate(rows):
            self.by_bucket[r["bucket_key"]].append(idx)

        self.input_block_keys = [k for k in self.by_bucket if len(k) == 3 and k[0] == "format" and k[1] == "input_block"]

    def set_epoch(self, epoch):
        self.epoch = epoch

    def _has_two_canonical_groups(self, batch):
        return len({self.rows[i]["canonical_id"] for i in batch}) >= 2

    def _sample_cross_n_input_blocks(self, key, core_len, rng):
        if INPUT_BLOCK_EXTRAS <= 0 or core_len <= 0:
            return []
        extra_count = min(INPUT_BLOCK_EXTRAS, max(1, core_len // 4))
        other = []
        for k in self.input_block_keys:
            if k != key:
                other.extend(self.by_bucket[k])
        rng.shuffle(other)
        return other[:extra_count]

    def __iter__(self):
        rng = random.Random(self.seed + self.epoch)
        batches = []

        for key, idxs0 in self.by_bucket.items():
            idxs = idxs0[:]
            rng.shuffle(idxs)

            is_input_block = len(key) == 3 and key[0] == "format" and key[1] == "input_block"
            if is_input_block:
                bucket_batch_size = max(2, self.batch_size - INPUT_BLOCK_EXTRAS)
                for start in range(0, len(idxs), bucket_batch_size):
                    core = idxs[start:start + bucket_batch_size]
                    batch = core + self._sample_cross_n_input_blocks(key, len(core), rng)
                    if self._has_two_canonical_groups(batch):
                        batches.append(batch)
            else:
                for start in range(0, len(idxs), self.batch_size):
                    batch = idxs[start:start + self.batch_size]
                    if self._has_two_canonical_groups(batch):
                        batches.append(batch)

        rng.shuffle(batches)
        for b in batches:
            yield b

    def __len__(self):
        return sum(1 for _ in self.__iter__())

class PairModel(nn.Module):
    def __init__(self, lm, pooling="last"):
        super().__init__()
        assert pooling in {"last", "mean"}
        self.lm = lm
        self.pooling = pooling

    def encode_with_ce(self, enc):
        out = self.lm(
            input_ids=enc["input_ids"],
            attention_mask=enc["attention_mask"],
            labels=enc["labels"],
            output_hidden_states=True,
            return_dict=True,
        )
        h = out.hidden_states[-1]

        if self.pooling == "last":
            last_idx = enc["attention_mask"].sum(dim=1) - 1
            batch_idx = torch.arange(h.size(0), device=h.device)
            z = h[batch_idx, last_idx]
        else:
            mask = enc["attention_mask"].unsqueeze(-1).to(h.dtype)
            z = (h * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)

        z = F.normalize(z, p=2, dim=-1)
        return z, out.loss

    def forward(self, enc_en, enc_py):
        z_en, ce_en = self.encode_with_ce(enc_en)
        z_py, ce_py = self.encode_with_ce(enc_py)
        ce_loss = 0.5 * (ce_en + ce_py)
        return z_en, z_py, ce_loss

def move_enc(enc, device):
    return {k: v.to(device) for k, v in enc.items()}

def build_model(pooling="last"):
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    tok.padding_side = "right"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    dtype = torch.float16 if DEVICE == "cuda" else torch.float32
    lm = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype)
    lm.config.use_cache = False
    lm.gradient_checkpointing_enable()

    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES,
        bias="none",
    )
    lm = get_peft_model(lm, lora_cfg)
    lm.enable_input_require_grads()
    lm.print_trainable_parameters()

    return PairModel(lm, pooling=pooling).to(DEVICE), tok

def _id_equal_mask(ids, device):
    n = len(ids)
    mask = torch.zeros((n, n), dtype=torch.bool, device=device)
    for i in range(n):
        for j in range(n):
            if ids[i] == ids[j]:
                mask[i, j] = True
    return mask

def _allowed_mask_from_ids(canonical_ids, equiv_ids, device):
    pos = _id_equal_mask(canonical_ids, device)
    same_equiv = _id_equal_mask(equiv_ids, device)
    allowed = pos | (~same_equiv)
    return allowed, pos

def _multi_positive_ce(sim, pos_mask, allowed_mask):
    very_neg = torch.finfo(sim.dtype).min
    sim_allowed = sim.masked_fill(~allowed_mask, very_neg)
    sim_pos = sim.masked_fill(~pos_mask, very_neg)
    log_num = torch.logsumexp(sim_pos, dim=1)
    log_den = torch.logsumexp(sim_allowed, dim=1)
    return -(log_num - log_den).mean()

def contrastive_loss(z_en, z_py, canonical_ids, equiv_ids, contrastive_ok, temperature):
    keep = torch.tensor(contrastive_ok, dtype=torch.bool, device=z_en.device)

    if keep.sum().item() < 2:
        zero = z_en.new_zeros(())
        return zero, None, None, None

    can = [cid for cid, ok in zip(canonical_ids, contrastive_ok) if ok]
    eqv = [eid for eid, ok in zip(equiv_ids, contrastive_ok) if ok]
    if len(set(can)) < 2:
        zero = z_en.new_zeros(())
        return zero, None, None, None

    z_en_k = z_en[keep]
    z_py_k = z_py[keep]
    sim = (z_en_k @ z_py_k.T) / temperature
    allowed, pos_mask = _allowed_mask_from_ids(can, eqv, sim.device)

    if ((allowed & ~pos_mask).sum().item()) == 0:
        zero = z_en.new_zeros(())
        return zero, sim, pos_mask, allowed

    loss_en_to_py = _multi_positive_ce(sim, pos_mask, allowed)
    loss_py_to_en = _multi_positive_ce(sim.T, pos_mask.T, allowed.T)
    return 0.5 * (loss_en_to_py + loss_py_to_en), sim, pos_mask, allowed

def total_loss(ce_loss, contrastive_loss, contrastive_lambda, contrastive_scale):
    ratio = ce_loss.detach() / contrastive_loss.detach().clamp_min(1e-8)
    weight = torch.clamp(ratio, max=contrastive_lambda)
    total = ce_loss + contrastive_scale * weight * contrastive_loss
    return total, weight * contrastive_scale

@torch.no_grad()
def evaluate(model, rows, tok):
    model.eval()
    collator = PairCollator(tok)
    loader = DataLoader(PairDataset(rows), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)

    all_en, all_py, all_semantic, all_canonical, all_equiv, all_ok = [], [], [], [], [], []
    ce_total = 0.0
    ce_n = 0

    for batch in loader:
        z_en, z_py, ce = model(move_enc(batch["en"], DEVICE), move_enc(batch["py"], DEVICE))
        bsz = len(batch["semantic_ids"])
        all_en.append(z_en.cpu())
        all_py.append(z_py.cpu())
        all_semantic.extend(batch["semantic_ids"])
        all_canonical.extend(batch["canonical_ids"])
        all_equiv.extend(batch["equiv_ids"])
        all_ok.extend(batch["contrastive_ok"])
        ce_total += ce.item() * bsz
        ce_n += bsz

    z_en = torch.cat(all_en, dim=0)
    z_py = torch.cat(all_py, dim=0)

    ok_idx = [i for i, ok in enumerate(all_ok) if ok]
    if len(ok_idx) >= 2:
        z_en_ok = z_en[ok_idx]
        z_py_ok = z_py[ok_idx]
        sem_ok = [all_semantic[i] for i in ok_idx]
        can_ok = [all_canonical[i] for i in ok_idx]
        eqv_ok = [all_equiv[i] for i in ok_idx]

        sim = z_en_ok @ z_py_ok.T
        allowed, pos_mask = _allowed_mask_from_ids(can_ok, eqv_ok, sim.device)
        sim_rank = sim.masked_fill(~allowed, torch.finfo(sim.dtype).min)
        top1 = sim_rank.argmax(dim=1)
        canonical_top1 = sum(can_ok[i] == can_ok[top1[i].item()] for i in range(len(can_ok))) / len(can_ok)

        pos_mask = pos_mask.cpu()
        allowed = allowed.cpu()
        neg_mask = allowed & ~pos_mask
        mean_pos = masked_mean(sim, pos_mask)
        mean_neg = masked_mean(sim, neg_mask)
    else:
        canonical_top1 = float("nan")
        mean_pos = float("nan")
        mean_neg = float("nan")

    return {
        "top1_canonical": canonical_top1,
        "mean_pos_sim": mean_pos,
        "mean_neg_sim": mean_neg,
        "ce": ce_total / ce_n,
        "contrastive_rows": len(ok_idx),
        "ce_only_rows": len(all_ok) - len(ok_idx),
    }

def print_token_length_summary(rows, tok):
    lens_en = [len(tok(r["text_en"], add_special_tokens=True)["input_ids"]) for r in rows]
    lens_py = [len(tok(r["text_py"], add_special_tokens=True)["input_ids"]) for r in rows]

    def summarize(xs):
        xs_sorted = sorted(xs)
        q95 = xs_sorted[int(0.95 * (len(xs_sorted) - 1))]
        return {
            "min": min(xs),
            "max": max(xs),
            "mean": sum(xs) / len(xs),
            "p95": q95,
            "over_max_length": sum(x > MAX_LENGTH for x in xs),
        }

    print("\nToken length summary before truncation:")
    print("EN:", summarize(lens_en))
    print("PY:", summarize(lens_py))

def preview(rows, n=12):
    print("\nPreview examples:")
    for r in rows[:n]:
        print("-" * 70)
        print("semantic_id :", r["semantic_id"])
        print("canonical_id:", r["canonical_id"])
        print("equiv_id    :", r["equiv_id"])
        print("bucket_key  :", r["bucket_key"])
        print("EN:", r["text_en"])
        print("PY:", r["text_py"])

def print_dataset_summary(rows):
    print("\nDataset summary:")
    print("rows:", len(rows))
    print("canonical groups:", len({r["canonical_id"] for r in rows}))

    expr_rows = [r for r in rows if r["semantic_id"].startswith("expr|")]
    rule_rows = [r for r in rows if r["semantic_id"].startswith("rule|")]
    format_rows = [r for r in rows if r["meta"].get("kind") == "format"]
    role_confusion_rows = [r for r in rows if r["meta"].get("role_confusion_view")]
    same_var = [r for r in rows if r["meta"].get("same_variable")]
    contrastive_rows = [r for r in rows if r["contrastive_ok"]]
    ce_only_rows = [r for r in rows if not r["contrastive_ok"]]

    print("expr rows:", len(expr_rows))
    print("rule rows:", len(rule_rows))
    print("format rows:", len(format_rows))
    print("role-confusion training views:", len(role_confusion_rows))
    print("same-variable binary rows:", len(same_var))
    print("contrastive rows:", len(contrastive_rows))
    print("CE-only rows:", len(ce_only_rows))

    multi_equiv_groups = Counter(r["equiv_id"] for r in rows)
    multi_equiv_count = sum(1 for _, c in multi_equiv_groups.items() if c > 1)
    print("truth/equiv groups with multiple surfaces:", multi_equiv_count)

def print_bucket_summary(rows):
    by_bucket = defaultdict(list)
    for r in rows:
        by_bucket[r["bucket_key"]].append(r)
    print("\nBucket summary:")
    for k, rs in sorted(by_bucket.items(), key=lambda kv: str(kv[0])):
        n_groups = len({r["canonical_id"] for r in rs})
        ok_groups = len({r["canonical_id"] for r in rs if r["contrastive_ok"]})
        ce_only = sum(1 for r in rs if not r["contrastive_ok"])
        print(f"{len(rs):4d} rows | {n_groups:4d} groups | {ok_groups:4d} ctr groups | {ce_only:3d} CE-only | {k}")

def print_same_variable_groups(rows, limit=8):
    print("\nSame-variable contrastive examples:")
    groups = defaultdict(list)
    for r in rows:
        if r["meta"].get("same_variable"):
            if r["semantic_id"].startswith("expr|"):
                groups[r["bucket_key"]].append(r)

    shown = 0
    for key, rs in sorted(groups.items()):
        if shown >= limit:
            break
        print("-" * 70)
        print("bucket_key:", key)
        for r in rs[:6]:
            print("canonical_id:", r["canonical_id"])
            print("equiv_id    :", r["equiv_id"])
            print("EN:", r["text_en"], " | PY:", r["text_py"])
        shown += 1

def rows_summary(rows):
    counts = Counter()
    for r in rows:
        if r["meta"].get("role_confusion_view"):
            counts["role_confusion_views"] += 1
            counts[f"role_confusion_{r['meta'].get('confusion_type', 'unknown')}"] += 1

        if r["semantic_id"].startswith("expr|"):
            counts["expr"] += 1
        elif r["semantic_id"].startswith("rule|"):
            counts["rule"] += 1
        elif r["meta"].get("kind") == "format":
            counts["format"] += 1
            counts[f"format_{r['meta'].get('format_type', 'unknown')}"] += 1
        else:
            counts["other"] += 1

    counts["total"] = len(rows)
    counts["canonical_groups"] = len({r["canonical_id"] for r in rows})
    counts["contrastive_rows"] = sum(1 for r in rows if r["contrastive_ok"])
    return dict(counts)

def apply_train_oversampling(train_rows, cfg):
    weight = int(cfg.get("format_row_weight", 1))
    if weight <= 1:
        return train_rows

    out = list(train_rows)
    dup_idx = 0
    for r in train_rows:
        is_format = r["meta"].get("kind") == "format"
        is_role_confusion_view = bool(r["meta"].get("role_confusion_view"))
        is_generated_full_prompt = r["meta"].get("format_type") == "full_prompt"
        if not is_format or is_role_confusion_view or is_generated_full_prompt:
            continue
        for k in range(weight - 1):
            rr = deepcopy(r)
            dup_idx += 1
            rr["semantic_id"] = f"{rr['semantic_id']}|train_dup={dup_idx}"
            out.append(rr)
    return out

def write_json(path, obj):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, sort_keys=True)

def append_jsonl(path, obj):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "a") as f:
        f.write(json.dumps(obj, sort_keys=True) + "\n")

def save_rows_preview(rows, path, n=40):
    preview_rows = []
    for r in rows[:n]:
        preview_rows.append({
            "semantic_id": r["semantic_id"],
            "canonical_id": r["canonical_id"],
            "equiv_id": r["equiv_id"],
            "bucket_key": str(r["bucket_key"]),
            "text_en": r["text_en"],
            "text_py": r["text_py"],
            "meta": r["meta"],
        })
    write_json(path, preview_rows)

DUP_KEY = "canonical_id"

def row_key(r):
    return r[DUP_KEY]

def key_set(rows):
    return {row_key(r) for r in rows}

def overlap_keys(a_rows, b_rows):
    return key_set(a_rows) & key_set(b_rows)

def drop_train_overlaps(train_rows, val_rows):
    val_keys = key_set(val_rows)
    kept = [r for r in train_rows if row_key(r) not in val_keys]
    skipped = len(train_rows) - len(kept)
    print(f"[bridge] skipped train rows overlapping val by {DUP_KEY}: {skipped}")
    return kept, skipped

def split_overlap_count(train_rows, val_rows):
    return len(overlap_keys(train_rows, val_rows))

def debug_dup_key():
    x = expr_canonical_id(Bin("and", Atom("a"), Atom("b")))
    y = expr_canonical_id(Bin("and", Atom("b"), Atom("a")))
    print("[bridge] duplicate key:", DUP_KEY)
    print("[bridge] order-sensitive check and(a,b) != and(b,a):", x != y, "|", x, "|", y)

def train_one_variant(cfg):
    cfg = dict(cfg)
    cfg["data_seed"] = DATA_SEED
    cfg["split_seed"] = SPLIT_SEED
    cfg["train_seed"] = TRAIN_SEED_BASE + int(cfg["idx"])

    variant_dir = os.path.join(SUITE_DIR, cfg["name"])
    best_dir = os.path.join(variant_dir, "best")
    last_dir = os.path.join(variant_dir, "last")
    summary_path = os.path.join(variant_dir, "summary.json")
    history_path = os.path.join(variant_dir, "metrics_history.jsonl")

    if RESUME_SKIP_COMPLETED and os.path.exists(summary_path) and os.path.exists(os.path.join(best_dir, "adapter_config.json")):
        print(f"\n[skip completed] {cfg['name']} -> {variant_dir}")
        with open(summary_path) as f:
            return json.load(f)

    os.makedirs(variant_dir, exist_ok=True)
    if os.path.exists(history_path):
        os.remove(history_path)

    print("\n" + "=" * 90)
    print(f"START VARIANT {cfg['idx']}: {cfg['name']}")
    print("=" * 90)
    print(json.dumps(cfg, indent=2, sort_keys=True))
    write_json(os.path.join(variant_dir, "config.json"), cfg)

    set_seed(cfg["train_seed"])

    base_cfg = dict(cfg)
    base_cfg["full_prompt_rows"] = 0
    base_cfg["seed"] = DATA_SEED

    base_rows = build_rows(base_cfg)
    full_prompt_val_rows = build_full_prompt_val_rows()
    forced_val_ids = important_format_val_ids(base_rows)
    train_rows_raw, val_rows = split_rows(base_rows, seed=SPLIT_SEED, force_val_ids=forced_val_ids)
    val_rows = val_rows + full_prompt_val_rows

    base_split_overlap = overlap_keys(train_rows_raw, val_rows)

    print(f"[bridge] base train/val {DUP_KEY} overlap before skip: {len(base_split_overlap)}")
    train_rows_raw, base_skipped_train_overlap = drop_train_overlaps(train_rows_raw, val_rows)

    base_train_ids = key_set(train_rows_raw)
    base_val_ids = key_set(val_rows)

    extra_full_prompt_rows = []
    n_full = int(cfg.get("full_prompt_rows", 0))
    if n_full > 0:
        extra_full_prompt_rows = build_full_prompt_rows(n_full, seed=DATA_SEED)
        train_rows_raw = train_rows_raw + extra_full_prompt_rows

    rows = base_rows + full_prompt_val_rows + extra_full_prompt_rows

    role_confusion_rows = build_role_confusion_views(train_rows_raw) if USE_EXTRA_ROWS else []
    train_rows_raw_with_confusion = train_rows_raw + role_confusion_rows
    train_rows = apply_train_oversampling(train_rows_raw_with_confusion, cfg)

    train_rows, final_skipped_train_overlap = drop_train_overlaps(train_rows, val_rows)

    split_overlap = overlap_keys(train_rows, val_rows)

    train_ids = key_set(train_rows)
    val_ids = key_set(val_rows)

    print(f"[bridge] final train/val {DUP_KEY} overlap after skip: {len(split_overlap)}")
    assert not split_overlap

    print(
        f"base_rows={len(base_rows)} | train_raw={len(train_rows_raw)} | "
        f"extra_full_prompt_train_only={len(extra_full_prompt_rows)} | "
        f"role_confusion_views={len(role_confusion_rows)} | "
        f"train_after_confusion_and_oversample={len(train_rows)} | val={len(val_rows)} | "
        f"forced_format_val_ids={len(forced_val_ids)} | full_prompt_val={len(full_prompt_val_rows)}"
    )
    print("rows summary:", rows_summary(rows))
    print("train_raw summary:", rows_summary(train_rows_raw))
    print("train summary:", rows_summary(train_rows))
    print("val summary:", rows_summary(val_rows))

    save_rows_preview(rows, os.path.join(variant_dir, "rows_preview.json"), n=50)
    save_rows_preview(val_rows, os.path.join(variant_dir, "val_rows_preview.json"), n=80)
    save_rows_preview(role_confusion_rows, os.path.join(variant_dir, "role_confusion_rows_preview.json"), n=80)

    model, tok = build_model(pooling=cfg["pooling"])

    collator = PairCollator(tok)
    train_sampler = PairBatchSampler(train_rows, BATCH_SIZE, cfg["train_seed"])
    train_loader = DataLoader(PairDataset(train_rows), batch_sampler=train_sampler, collate_fn=collator)
    steps_per_epoch = sum(1 for _ in train_sampler)
    train_sampler.set_epoch(0)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = math.ceil(steps_per_epoch / GRAD_ACCUM) * EPOCHS
    warmup_steps = int(WARMUP_RATIO * total_steps)
    sch = get_linear_schedule_with_warmup(opt, warmup_steps, total_steps)

    best_score = float("inf")
    best_metrics = None
    best_epoch = None
    start_time = time.time()

    for epoch in range(EPOCHS):
        train_sampler.set_epoch(epoch)
        model.train()
        opt.zero_grad(set_to_none=True)

        run_total = run_ce = run_ctr = run_w = 0.0
        run_n = 0

        for step, batch in enumerate(train_loader):
            z_en, z_py, ce = model(move_enc(batch["en"], DEVICE), move_enc(batch["py"], DEVICE))
            ctr, _, _, _ = contrastive_loss(
                z_en,
                z_py,
                batch["canonical_ids"],
                batch["equiv_ids"],
                batch["contrastive_ok"],
                temperature=float(cfg["temperature"]),
            )
            loss, weight = total_loss(
                ce,
                ctr,
                contrastive_lambda=BASE_LAMBDA,
                contrastive_scale=float(cfg["contrastive_scale"]),
            )
            (loss / GRAD_ACCUM).backward()

            if (step + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                opt.step()
                sch.step()
                opt.zero_grad(set_to_none=True)

            bsz = len(batch["semantic_ids"])
            run_total += loss.item() * bsz
            run_ce += ce.item() * bsz
            run_ctr += ctr.item() * bsz
            run_w += weight.item() * bsz
            run_n += bsz

        metrics = evaluate(model, val_rows, tok)
        score = metrics["ce"]

        row = {
            "variant": cfg["name"],
            "epoch": epoch + 1,
            "train_total": run_total / run_n,
            "train_ce": run_ce / run_n,
            "train_ctr": run_ctr / run_n,
            "ctr_weight": run_w / run_n,
            **{f"val_{k}": v for k, v in metrics.items()},
        }
        append_jsonl(history_path, row)

        print(
            f"[{cfg['name']} epoch {epoch+1:03d}/{EPOCHS}] "
            f"train_total={row['train_total']:.4f} | "
            f"train_ce={row['train_ce']:.4f} | "
            f"train_ctr={row['train_ctr']:.4f} | "
            f"ctr_weight={row['ctr_weight']:.2f} | "
            f"val_top1_canon={metrics['top1_canonical']:.4f} | "
            f"val_pos={metrics['mean_pos_sim']:.4f} | "
            f"val_neg={metrics['mean_neg_sim']:.4f} | "
            f"val_ce={metrics['ce']:.4f}"
        )

        if score < best_score:
            best_score = score
            best_metrics = deepcopy(metrics)
            best_epoch = epoch + 1
            os.makedirs(best_dir, exist_ok=True)
            model.lm.save_pretrained(best_dir, safe_serialization=True)
            tok.save_pretrained(best_dir)
            print("saved best adapter by val_ce to:", best_dir)

    os.makedirs(last_dir, exist_ok=True)
    model.lm.save_pretrained(last_dir, safe_serialization=True)
    tok.save_pretrained(last_dir)

    elapsed_min = (time.time() - start_time) / 60.0
    summary = {
        "config": cfg,
        "variant_dir": variant_dir,
        "best_dir": best_dir,
        "last_dir": last_dir,
        "best_epoch": best_epoch,
        "best_score": best_score,
        "best_metrics": best_metrics,
        "rows_summary": rows_summary(rows),
        "train_raw_summary": rows_summary(train_rows_raw),
        "extra_full_prompt_train_only_summary": rows_summary(extra_full_prompt_rows),
        "role_confusion_summary": rows_summary(role_confusion_rows),
        "train_after_confusion_and_oversample_summary": rows_summary(train_rows),
        "val_summary": rows_summary(val_rows),
        "split_protection": {
            "unit": "canonical_id",
            "base_train_val_overlap_before_skip": len(base_split_overlap),
            "base_skipped_train_overlap": base_skipped_train_overlap,
            "final_skipped_train_overlap": final_skipped_train_overlap,
            "final_train_val_overlap_after_skip": len(split_overlap),
            "base_train_canonical_ids": len(base_train_ids),
            "base_val_canonical_ids": len(base_val_ids),
            "final_train_canonical_ids": len(train_ids),
            "final_val_canonical_ids": len(val_ids),
        },
        "selection_rule": "best adapter is selected by lowest validation ce",
        "forced_format_val_ids": len(forced_val_ids),
        "full_prompt_val_summary": rows_summary(full_prompt_val_rows),
        "elapsed_minutes": elapsed_min,
    }
    write_json(summary_path, summary)
    print("\nDONE:", cfg["name"])
    print("best_epoch:", best_epoch)
    print("best_metrics:", best_metrics)
    print("saved best adapter to:", best_dir)
    print("saved last adapter to:", last_dir)

    del model, tok, opt, sch, train_loader, train_sampler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary

def write_suite_summary(summaries):
    out_json = os.path.join(SUITE_DIR, "summary_all.json")
    write_json(out_json, summaries)

    out_csv = os.path.join(SUITE_DIR, "summary_all.csv")
    fields = [
        "idx", "name", "full_prompt_rows", "format_row_weight",
        "contrastive_scale", "temperature", "pooling",
        "best_epoch", "best_score", "top1_canonical",
        "mean_pos_sim", "mean_neg_sim", "ce",
        "best_dir", "last_dir",
    ]
    with open(out_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for s in summaries:
            cfg = s.get("config", {})
            m = s.get("best_metrics") or {}
            writer.writerow({
                "idx": cfg.get("idx"),
                "name": cfg.get("name"),
                "full_prompt_rows": cfg.get("full_prompt_rows"),
                "format_row_weight": cfg.get("format_row_weight"),
                "contrastive_scale": cfg.get("contrastive_scale"),
                "temperature": cfg.get("temperature"),
                "pooling": cfg.get("pooling"),
                "best_epoch": s.get("best_epoch"),
                "best_score": s.get("best_score"),
                "top1_canonical": m.get("top1_canonical"),
                "mean_pos_sim": m.get("mean_pos_sim"),
                "mean_neg_sim": m.get("mean_neg_sim"),
                "ce": m.get("ce"),
                "best_dir": s.get("best_dir"),
                "last_dir": s.get("last_dir"),
            })

    print("\nSuite summary saved:")
    print(out_json)
    print(out_csv)

def train_suite():
    summaries = []
    selected = [v for v in VARIANTS if v["idx"] in RUN_IDS]

    debug_dup_key()

    for i, cfg in enumerate(selected):
        summaries.append(train_one_variant(cfg))
        write_suite_summary(summaries)

    return summaries

summaries = train_suite()
summaries
